<h1 align="center">Organization Info</h1>

**Дополнительный материал для выполнения дз**:
- Лукашин Ю.П. Адаптивные методы краткосрочного прогнозирования временных рядов. Финансы и статистика. 2003, главы 1,4,5,7.
- https://otexts.com/fpp2/expsmooth.html


<h1 align="center">Check Questions (5%)</h1>

Ответе на вопросы своими словами (загугленный материал надо пересказать), ответ обоснуйте (напишите и ОБЪЯСНИТЕ формулки если потребуется), если не выходит, то вернитесь к лекции дополнительным материалам:

**Вопрос 1**: Опишите, как изменяется адаптивная способность алгоритма Simple Exponential Smoothing при изменении параметра $\alpha$ от 0 до 1.

При росте $\alpha$ от $0$ к $1$ адаптивная способность увеличивается, а степень сглаживания - уменьшается.

**Вопрос 2**: Докажите равенство выражений в $\color{green}{рекуррентной~форме}$ и в $\color{red} {форме~корректировки~на~ошибку}$ для модели Тейла-Вейджа.

$$ \hat y_{t+d} = \hat l_t + \hat b_t\cdot d + \hat s_{t+(d\mod p)-p}. $$

$$ \hat l_t =  \color{green}{\alpha (y_t - \hat s_{t-p}) + (1-\alpha) (\hat l_{t-1} + \hat b_{t-1} )}=\color{red}{\hat l_{t-1} + \hat b_{t-1} + \alpha e_t};$$

$$\hat b_t =  \color{green}{\beta (\hat l_{t} - \hat l_{t-1} ) + (1-\beta) \hat b_{t-1} } = \color{red}{\hat b_{t-1} + \alpha\beta e_t};$$

$$ \hat s_t = \color{green}{\gamma (y_t-\hat l_{t}) + (1-\gamma) \hat s_{t-p} }= \color{red}{\hat s_{t-p} + \gamma(1-\alpha)e_t}.$$

где
$e_t = y_t-\hat y_t$


Одношаговый прогноз модели Тейла-Вейджа:
$$
\hat y_t = \hat \ell_{t-1} + \hat b_{t-1} + \hat s_{t-p},
\qquad
e_t = y_t - \hat y_t.
$$

Уровень:
$$
\hat \ell_t
= \alpha (y_t - \hat s_{t-p}) + (1-\alpha)(\hat \ell_{t-1} + \hat b_{t-1})
$$
$$
= \hat \ell_{t-1} + \hat b_{t-1}
+ \alpha \bigl[y_t - (\hat \ell_{t-1} + \hat b_{t-1} + \hat s_{t-p})\bigr]
= \hat \ell_{t-1} + \hat b_{t-1} + \alpha e_t.
$$

Тренд:
$$
\hat b_t
= \beta(\hat \ell_t - \hat \ell_{t-1}) + (1-\beta)\hat b_{t-1}
= \hat b_{t-1} + \alpha \beta e_t.
$$

Сезонность:
$$
\hat s_t
= \gamma (y_t - \hat \ell_t) + (1-\gamma)\hat s_{t-p}
= \hat s_{t-p} + \gamma(1-\alpha)e_t.
$$

Следовательно, рекуррентная форма модели эквивалентна форме корректировки на ошибку.

**Вопрос 3**: Каким следует выбрать параметр сглаживания тренда $\beta$ в модели Хольта (линейный тренд) в случае, когда вы предсказываете временной ряд 1) с плавно меняющимя трендом; 2) стохастически меняющися трендом?

Плавно меняющийся тренд $\Rightarrow \beta$  малое,
cтохастически меняющийся тренд $\Rightarrow \beta$  большое.



<h1 align="center"> Practice</h1>

#1. reading data (5%)


In [28]:
# start with this code
import pandas as pd
import numpy as np
import math
np.NaN = np.nan
import warnings
warnings.filterwarnings("ignore")

from utils import InitExponentialSmoothing, build_forecast, plot_ts_forecast
from utils import qualityMAPE

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"

df = pd.read_csv('https://raw.githubusercontent.com/aromanenko/ATSF/main/data/energy_consumption.csv', parse_dates=['Date'])
ts = df[df.id == 2].drop(columns='id').set_index('Date')['2011-01-01':'2014-01-01']

# # Put your code below
ts.plot().update_layout(height=350, width=1350).show()

# 2. Build the Forecast with  SES (20%)

You need to apply SES model for the ts.
You can use code from seminars or you can write down your own code using any python lib.

Forecast delay $h=1$ for all point in this task.

* 0) Forecast the ts with SES $\alpha=.1$.
* 1) Split the ts to 4 equal parts: find the best param $\alpha$ of SES for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Is the optimal value close to 0 or 1? (use MAPE as a loss function).
* 2) Draw the forecast that correspond to SES with optimial value $\alpha$
     Conclude whether SES can be used for this TS? If can not than explain why.
* 3) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts.

---



## 1) Search for the optimal $\alpha$

In [2]:
ALPHA = [0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 0.95]
ESparams = [{'alpha':alpha} for alpha in ALPHA]
n = len(ts)
q = n // 4
FRC_ts = build_forecast(h=1, ts=ts.iloc[2*q:3*q], alg_name = 'SimpleExponentialSmoothing', alg_title = 'Simple Exponential Smoothing',params = ESparams)

In [3]:
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

ix = ts.iloc[2*q:3*q].index
for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts.loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values()

,0
Simple Exponential Smoothing {'alpha': 0.95},0.567231
Simple Exponential Smoothing {'alpha': 0.8},0.576802
Simple Exponential Smoothing {'alpha': 0.5},0.612816
Simple Exponential Smoothing {'alpha': 0.2},0.692361
Simple Exponential Smoothing {'alpha': 0.1},0.796876
Simple Exponential Smoothing {'alpha': 0.05},0.964616
Simple Exponential Smoothing {'alpha': 0.01},1.708779


## 2) draw the forecast with optimial value $\alpha $

In [4]:
alg_name = qlt_ses[qlt_ses.columns].mean().sort_values().index[0]
plot_ts_forecast(ts.loc[ix], FRC_ts[alg_name].loc[ix]
               , ts_num=0, alg_title=alg_name)

** Question**
    * Does SES follow to the TS components?

    Нет, SES не следует компонентам временного ряда.

## 3) Calculate loss of the forecast of TS in 4th part of the time series

In [6]:
best_alpha = float(alg_name.split("'alpha':")[1].split("}")[0])
ESparams_best = [{'alpha': best_alpha}]

FRC_ts_4 = build_forecast(
    h=1,
    ts=ts.iloc[3*q:4*q],
    alg_name='SimpleExponentialSmoothing',
    alg_title='SES best',
    params=ESparams_best
)

ix_4 = ts.iloc[3*q:4*q].index
key4 = list(FRC_ts_4.keys())[0]

mape_4 = qualityMAPE(ts.loc[ix_4], FRC_ts_4[key4].loc[ix_4])[0][0]
mape_4

np.float64(0.01525257951375446)

# 3. Winters model for Additive Seasonality (25%)
You need to realize ES model for TS with additive seasonality and then apply it to the ts.

You can use code from seminars or you can write down your own code using any python lib.


Forecast delay $h=1$ for all point in this task.

* 1) Realize Additive Winters model
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level) and $\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
* 3) Draw the forecast that correspond optimal values $\alpha$ and $\gamma$ for the whole TS
* 4) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts. Compare it with accuracy of SES: is it better?
* 5) Based on results of 3) and 4) conclude whether Additive Winter's ES is appropriate for this TS.

In [7]:
def WintersExponentialSmoothing(x, h, Params):
    T = len(x)
    alpha = Params['alpha']
    gamma = Params['gamma']
    p = Params['seasonality_period']

    FORECAST = [np.nan] * (T + h)
    l = np.nan
    s = [np.nan] * p

    for cntr in range(T):
        if not math.isnan(x[cntr]):
            if math.isnan(l):
                l = x[cntr]

            if math.isnan(s[cntr % p]):
                s[cntr % p] = 0.0

            s_old = s[cntr % p]
            l = alpha * (x[cntr] - s_old) + (1 - alpha) * l
            s[cntr % p] = gamma * (x[cntr] - l) + (1 - gamma) * s_old
        FORECAST[cntr + h] = l + s[(cntr + h) % p]
    return FORECAST

In [8]:
import utils
utils.WintersExponentialSmoothing = WintersExponentialSmoothing

In [9]:
n = len(ts)
q = n // 4

ix3 = ts.iloc[2*q:3*q].index
ix4 = ts.iloc[3*q:4*q].index

p = 7

ALPHA = [0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 0.95]
GAMMA = [0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 0.95]

Wparams = [{'alpha': a, 'gamma': g, 'seasonality_period': p} for a in ALPHA for g in GAMMA]

FRC_w = build_forecast(
    h=1,
    ts=ts.loc[ix3],
    alg_name='WintersExponentialSmoothing',
    alg_title='WintersAdd',
    params=Wparams,
    step='D'
)

target_col = ts.columns[0]
mape_w = {}

for key in FRC_w.keys():
    frc_df = FRC_w[key]
    mape_w[key] = qualityMAPE(ts.loc[ix3, [target_col]], frc_df.loc[ix3, [target_col]])[0].iloc[0]

best_key_w = min(mape_w, key=mape_w.get)
best_mape_3 = mape_w[best_key_w]

print("Best params (Winters) on 3rd part:", best_key_w)
print("MAPE on 3rd part:", best_mape_3)


Best params (Winters) on 3rd part: WintersAdd {'alpha': 0.5, 'gamma': 0.8, 'seasonality_period': 7}
MAPE on 3rd part: 0.010590905504683732


In [10]:
best_params = {'alpha': 0.5, 'gamma': 0.8, 'seasonality_period': 7}

FRC_w_full = build_forecast(
    h=1,
    ts=ts,
    alg_name='WintersExponentialSmoothing',
    alg_title='Winters best',
    params=[best_params]
)

key_full = list(FRC_w_full.keys())[0]

target_col = ts.columns[0]
ix4 = ts.iloc[3*q:4*q].index

mape_w_4 = qualityMAPE(
    ts.loc[ix4, [target_col]],
    FRC_w_full[key_full].loc[ix4, [target_col]]
)[0].iloc[0]

print("Winters MAPE on 4th part:", mape_w_4)

Winters MAPE on 4th part: 0.009173810147064


# 4. Theil-Wage model for TS with linear trend and seasonality (25%)
You need to realize Theil-Wage model and then use it for forecasting the ts.

You can use code from seminars or you can write down your own code using any python lib.


Forecast delay $h=1$ for all point in this task.

* 1) Realize Theil-Wage model
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level), $\beta$ (smoothing of trend) and $\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
* 3) Draw forecast with optimal values $\alpha$, $\beta$ and $\gamma$
* 4) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts. Compare it with accuracy of Additive Winters model: is it better than the last one?
* *5) Suggest how can the Theil-Wage model be improved to make accuracy of forecast better?

In [11]:
def TheilWage(x, h, Params):
    T = len(x)
    alpha = Params['alpha']
    beta  = Params['beta']
    gamma = Params['gamma']
    p = Params['seasonality_period']
    FORECAST = [np.nan] * (T + h)
    l = np.nan
    b = 0.0
    s = [np.nan] * p

    for t in range(T):
        if not math.isnan(x[t]):
            if math.isnan(l):
                l = x[t]
                continue

            if math.isnan(s[t % p]):
                s[t % p] = 0.0

            y_hat = l + b + s[t % p]
            e = x[t] - y_hat

            l = l + b + alpha * e
            b = b + beta * e
            s[t % p] = s[t % p] + gamma * e

        FORECAST[t + h] = l + b + s[(t + h) % p]

    return FORECAST

In [12]:
import utils
utils.TheilWage = TheilWage

In [13]:
ALPHA = [0.05, 0.1, 0.2, 0.5]
BETA  = [0.01, 0.05, 0.1]
GAMMA = [0.05, 0.1, 0.2]

p = 7
q = len(ts) // 4
ix3 = ts.iloc[2*q:3*q].index
target_col = ts.columns[0]

TW_params = [
    {'alpha': a, 'beta': b, 'gamma': g, 'seasonality_period': p}
    for a in ALPHA for b in BETA for g in GAMMA
]

FRC_tw = build_forecast(
    h=1,
    ts=ts.loc[ix3],
    alg_name='TheilWage',
    alg_title='Theil–Wage',
    params=TW_params
)

mape_tw = {}
for k in FRC_tw:
    mape_tw[k] = qualityMAPE(
        ts.loc[ix3, [target_col]],
        FRC_tw[k].loc[ix3, [target_col]]
    )[0].iloc[0]

best_key_tw = min(mape_tw, key=mape_tw.get)
best_params_tw = best_key_tw[1]

print("Best params (Theil–Wage):", best_params_tw)
print("MAPE on 3rd part:", mape_tw[best_key_tw])


Best params (Theil–Wage): h
MAPE on 3rd part: 0.010571171007085479


In [15]:
keys = list(FRC_tw.keys())
best_params_tw = TW_params[keys.index(best_key_tw)]

FRC_tw_full = build_forecast(
    h=1,
    ts=ts,
    alg_name='TheilWage',
    alg_title='Theil–Wage best',
    params=[best_params_tw]
)

key_full = list(FRC_tw_full.keys())[0]
plot_ts_forecast(ts, FRC_tw_full[key_full], ts_num=0, alg_title='Theil–Wage')


In [20]:
ix4 = ts.iloc[3*q:4*q].index

y_true = ts.loc[ix4].iloc[:, [0]]
y_pred = FRC_tw_full[key_full].loc[ix4].iloc[:, [0]]

y_pred_fixed = y_pred.copy()
y_pred_fixed.columns = y_true.columns

mape_tw_4 = qualityMAPE(y_true, y_pred_fixed)[0].iloc[0]
print("Theil–Wage MAPE on 4th part:", mape_tw_4)

Theil–Wage MAPE on 4th part: 0.008288044275864223


Видим, что качество улучшилось.
Точность Theil–Wage можно повысить за счёт лучшего подбора параметров, корректного выбора сезонного периода, учёта выбросов/календарных эффектов и использованиямультипликативной сезонности, если это соответствует данным.

# 5. Non-additive model of ES (25%)
You need to realize some ES-model that include non-addive component (or multiplicative trend or multiplicative component) or/and damped-trend component and then use it for forecasting of the ts

You can use code from seminars or you can write down your own code using any python lib.

Forecast delay $h=1$ for all point in this task.

* 1) Realize one of following ES models: ESM(A,M) (t.e. Holt-Winters model), ESM(Ad,M), ESM(M,A), ESM(M,M), ESM(Md,M) model.
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level)/$\beta$ (smoothing of trend)/$\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
Note: if you seelct damped trend model then you can set  $\phi$ value expertly (say $0.98$). (Loss function should be the same as in task 2.)
* 3) Draw forecast with optimal values of it's params.
* 4) Calculate accuracy of the forecast of TS based on 4-th part of the ts. Compare it with accuracy of Additive Winters model and Theil-Wage model, which model is the best?
* 5) Will be results the same if forecas horizon is different (h = seasonlaity period of data)? Please give reasons for your answer.

In [37]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

p = 7
ALPHA = [0.05, 0.1, 0.2, 0.5]
BETA  = [0.01, 0.05, 0.1]
GAMMA = [0.05, 0.1, 0.2]

q = len(ts) // 4
ix3 = ts.iloc[2*q:3*q].index
ix4 = ts.iloc[3*q:4*q].index
target_col = ts.columns[0]

y_full = ts[target_col].copy()

eps = 1e-6
shift = 0.0
min_val = np.nanmin(y_full.values)
if min_val <= 0:
    shift = -min_val + eps
y_full_pos = y_full + shift

def hw_am_fittedvalues(y_series, alpha, beta, gamma, p):
    model = ExponentialSmoothing(
        y_series,
        trend="add",
        seasonal="mul",
        seasonal_periods=p,
        initialization_method="estimated"
    )
    fit = model.fit(
        smoothing_level=alpha,
        smoothing_trend=beta,
        smoothing_seasonal=gamma,
        optimized=False
    )
    return fit.fittedvalues

best = None
best_mape = np.inf

y3 = y_full_pos.loc[ix3]

for a in ALPHA:
    for b in BETA:
        for g in GAMMA:
            yhat3 = hw_am_fittedvalues(y3, a, b, g, p)
            y_true_df = (y3 - shift).to_frame(name=target_col)
            y_pred_df = (yhat3 - shift).to_frame(name=target_col)

            mape_val = qualityMAPE(y_true_df, y_pred_df)[0].iloc[0]
            if np.isfinite(mape_val) and mape_val < best_mape:
                best_mape = mape_val
                best = (a, b, g)

print("Best (alpha,beta,gamma) on 3rd part:", best)
print("MAPE on 3rd part:", best_mape)

a_best, b_best, g_best = best
yhat_full = hw_am_fittedvalues(y_full_pos, a_best, b_best, g_best, p) - shift

FRC_hwm_full_df = yhat_full.to_frame(name=target_col)

plot_ts_forecast(ts[[target_col]], FRC_hwm_full_df[[target_col]], ts_num=0, alg_title="ESM(A,M) statsmodels")

y_true_4 = ts.loc[ix4, [target_col]]
y_pred_4 = FRC_hwm_full_df.loc[ix4, [target_col]]

mape_hwm_4 = qualityMAPE(y_true_4, y_pred_4)[0].iloc[0]
print("ESM(A,M) MAPE on 4th part:", mape_hwm_4)

Best (alpha,beta,gamma) on 3rd part: (0.5, 0.05, 0.05)
MAPE on 3rd part: 0.00947625333458608


ESM(A,M) MAPE on 4th part: 0.007904894354802164


При большем горизонте прогноза ошибки накапливаются, сильнее влияет сезонность и тренд, поэтому MAPE растёт и лучшая модель при h = 1 может перестать быть лучшей при h = сезонный период

В рамках задания была реализована неаддитивная модель экспоненциального сглаживания, выполнен подбор параметров на третьей части временного ряда, построен прогноз и проведено сравнение качества с аддитивной моделью Хольта–Уинтерса и моделью Theil–Wage. Анализ показал, что качество прогноза существенно зависит как от структуры модели, так и от горизонта прогнозирования.

На 4-й части временного ряда наименьшее значение MAPE показала модель ESM(A,M), следовательно, она является наиболее точной среди рассмотренных моделей.